In [1]:
import numpy as np
import os
from datasets import load_dataset
import subprocess
from pandas import DataFrame
from pathlib import Path
import json
import textwrap
import coverage

In [2]:
ds = load_dataset("dz1/CodeScore-MBPP-ET")
CodeScore_df = DataFrame(ds['train'][89:99])
CodeScore_df

,text,code,task_id,test_setup_code,test_list,challenge_test_list,entry_point
0,Write a python function to find the length of ...,def len_log(list1):\r\n max=len(list1[0])\r...,90,,"[assert len_log([""python"",""PHP"",""bigdata""]) ==...",[],len_log
1,Write a function to check if a substring is pr...,"def find_substring(str1, sub_str):\r\n if an...",91,,"[assert find_substring([""red"", ""black"", ""white...",[],find_substring
2,Write a function to check whether the given nu...,def is_undulating(n): \r\n\tif (len(n) <= 2): ...,92,,"[assert is_undulating(""1212121"") == True, asse...",[],is_undulating
3,Write a function to calculate the value of 'a'...,"def power(a,b):\r\n\tif b==0:\r\n\t\treturn 1\...",93,,"[assert power(3,4) == 81, assert power(2,3) ==...",[],power
4,Write a function to extract the index minimum ...,from operator import itemgetter \r\ndef index_...,94,,"[assert index_minimum([('Rash', 143), ('Manjee...",[],index_minimum
5,Write a python function to find the minimum le...,def Find_Min_Length(lst): \r\n minLength =...,95,,"[assert Find_Min_Length([[1],[1,2]]) == 1, ass...",[],Find_Min_Length
6,Write a python function to find the number of ...,def divisor(n):\r\n for i in range(n):\r\n ...,96,,"[assert divisor(15) == 4 , assert divisor(12) ...",[],divisor
7,Write a function to find frequency count of li...,def frequency_lists(list1):\r\n list1 = [it...,97,,"[assert frequency_lists([[1, 2, 3, 2], [4, 5, ...",[],frequency_lists
8,Write a function to multiply all the numbers i...,def multiply_num(numbers): \r\n total = 1\...,98,,"[assert multiply_num((8, 2, 3, -1, 7))==-67.2,...",[],multiply_num
9,Write a function to convert the given decimal ...,def decimal_to_binary(n): \r\n return bin(n...,99,,"[assert decimal_to_binary(8) == '1000', assert...",[],decimal_to_binary


In [6]:
import google.generativeai as genai

def gemini_generate(prompt_technique,problemrange):
    genai.configure(api_key=GOOGLE_API_KEY)
    model = genai.GenerativeModel("gemini-2.5-flash")

    for test_idx in range(problemrange):
        response = model.generate_content(CodeScore_df["text_"+prompt_technique][test_idx])
        print(response.text)

        # Step 1: Remove Markdown fences if present
        raw_response = response.text
        cleaned = raw_response.strip()
        if cleaned.startswith("```"):
            cleaned = "\n".join(cleaned.split("\n")[1:-1])  # remove first and last lines

        # Step 2: Parse JSON
        data = json.loads(cleaned)

        # Step 3: Extract the 'code' field
        code_text = data["code"]

        # Step 4: Write to file
        folder_path = Path("gemini_written_code/"+prompt_technique)
        folder_path.mkdir(parents=True, exist_ok=True)

        file_path = folder_path / (CodeScore_df['entry_point'][test_idx] + ".py")
        file_path.write_text(code_text, encoding="utf-8")

        print(f"Successfully wrote code to '{file_path}'")

In [10]:
# def test_suite_single_problem(model_name,prompt_technique,problem_idx, test_list):
#     folder_name= model_name+"_written_code/"+prompt_technique
#     folder_path = Path(folder_name)
#     problem_name = CodeScore_df['entry_point'][problem_idx]

#     file_name = f"{problem_name}.py"
#     file_path = folder_path / file_name

#     if not file_path.exists():
#         print(f"File not found: {file_path}")
#         return

#     with open(file_path, "r", encoding="utf-8") as f:
#         problem_code = f.read()

#     namespace = {}
#     try:
#         exec(problem_code, namespace)
#     except Exception as e:
#         print(f"Error executing code for {problem_name} in {folder_name}: {e}")
#         return

#     passed = 0
#     total = len(test_list)

#     for test in test_list:
#         try:
#             exec(test, namespace)
#             passed += 1
#         except Exception:
#             pass  # test failed

#     line_percent = -1
#     branch_percent = -1
#     test_percent = 100.0*(passed)/total

#     return {"problem_name": problem_name, "line_percent": line_percent, "branch_percent":branch_percent, "test_percent": test_percent }

def test_suite_single_problem(model_name, prompt_technique, problem_idx, test_list, CodeScore_df):
    folder_name = f"{model_name}_written_code/{prompt_technique}"
    folder_path = Path(folder_name)
    problem_name = CodeScore_df['entry_point'][problem_idx]

    file_name = f"{problem_name}.py"
    file_path = folder_path / file_name

    if not file_path.exists():
        print(f"File not found: {file_path}")
        return

    with open(file_path, "r", encoding="utf-8") as f:
        problem_code = f.read()

    # --- Start coverage with branch tracking ---
    cov = coverage.Coverage(branch=True)
    cov.start()

    namespace = {}

    try:
        # Compile with filename so coverage recognizes it
        compiled_code = compile(problem_code, str(file_path), "exec")
        exec(compiled_code, namespace)
    except Exception as e:
        print(f"Error executing code for {problem_name} in {folder_name}: {e}")
        cov.stop()
        return

    passed = 0
    total = len(test_list)

    # Run tests
    for test in test_list:
        try:
            exec(test, namespace)
            passed += 1
        except Exception:
            pass

    # Stop coverage collection
    cov.stop()
    cov.save()

    # --- Export coverage data as JSON ---
    json_path = Path("tmp_coverage.json")
    cov.json_report(outfile=str(json_path))

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Get per-file data
    file_data = data["files"].get(str(file_path))
    if not file_data:
        return {
            "problem": problem_name,
            "#passed": 0,
            "line%": 0.0,
            "branch%": 0.0,
            "test%": 0.0,
            "metric": abs(test_percent-branch_percent-line_percent)*test_percent/100
        }

    summary = file_data["summary"]

    # --- Extract metrics cleanly ---
    line_percent = summary.get("percent_covered", 0.0)
    covered_branches = summary.get("covered_branches", 0)
    total_branches = summary.get("num_branches", 0)
    branch_percent = (100.0 * covered_branches / total_branches) if total_branches else 0.0
    test_percent = 100.0 * passed / total if total > 0 else 0.0

    return {
        "problem": problem_name,
        "#passed": passed,
        "line%": line_percent,
        "branch%": branch_percent,
        "test%": test_percent,
        "metric": abs(test_percent-branch_percent-line_percent)*test_percent/100
    }
  
def test_suite(model_name, prompt_technique, CodeScore_df):
    folder_name= model_name+"_written_code/"+prompt_technique
    test_results = []
    
    for idx in range(len(CodeScore_df['entry_point'])):
        # Get the corresponding tests from the dataframe
        problem_test_list = CodeScore_df['test_list'][idx]

        test_result = test_suite_single_problem(model_name,prompt_technique,idx,problem_test_list, CodeScore_df)

        test_results.append(test_result)
        
    return test_results

def run_tests_visible(model_name,prompt_technique,problem_idx, test_list):
    entry_point = CodeScore_df['entry_point'][problem_idx]
    problem_test_list = test_list

    # Read the Python code from the file
    with open(model_name+"_written_code/"+prompt_technique+"/"+entry_point+".py", "r", encoding="utf-8") as f:
        problem_code = f.read()

    # Prepare namespace and execute the code
    namespace = {}
    exec(problem_code, namespace)

    for test in problem_test_list:
        try:
            exec(test, namespace)
            print(f"✅ Passed: {test}")
        except Exception as e:
            print(f"❌ Failed: {test}")
            print("   Error:", e)

In [7]:
def run_tests_visible(model_name,prompt_technique,problem_idx):
    entry_point = CodeScore_df['entry_point'][problem_idx]
    problem_test_list = CodeScore_df['test_list'][problem_idx]

    # Read the Python code from the file
    with open(model_name+"_written_code/"+prompt_technique+"/"+entry_point+".py", "r", encoding="utf-8") as f:
        problem_code = f.read()

    # Prepare namespace and execute the code
    namespace = {}
    exec(problem_code, namespace)

    for test in problem_test_list:
        try:
            exec(test, namespace)
            print(f"✅ Passed: {test}")
        except Exception as e:
            print(f"❌ Failed: {test}")
            print("   Error:", e)

# TEST EVERY TEST
def get_tests_passed(model_name,prompt_technique):
    folder_name= model_name+"_written_code/"+prompt_technique
    folder_path = Path(folder_name)
    problems_passed = []
    
    for idx, problem_name in enumerate(CodeScore_df['entry_point']):
        # Build the file path
        file_name = f"{problem_name}.py"
        file_path = folder_path / file_name

        if not file_path.exists():
            print(f"File not found: {file_path}")
            continue

        # Read the Python code from the file
        with open(file_path, "r", encoding="utf-8") as f:
            problem_code = f.read()

        # Get the corresponding tests from the dataframe
        problem_test_list = CodeScore_df['test_list'][idx]

        # Prepare namespace and execute the code
        namespace = {}
        try:
            exec(problem_code, namespace)
        except Exception as e:
            print(f"Error executing code for {problem_name} in {folder_name}: {e}")
            continue

        # Run tests
        passed = 0
        total = len(problem_test_list)

        for test in problem_test_list:
            try:
                exec(test, namespace)
                passed += 1
            except Exception:
                pass  # test failed
        
        if passed >= total: #passed all tests
            problems_passed.append(problem_name)

        print(f"{folder_name}/{file_name}: Passed {passed}/{total} tests")

    return problems_passed


## Generate Tests

In [37]:
def gemini_generate_automated_specifications(prompt_template, problem_statement, method_signature):
    import google.generativeai as genai

    formatted_prompt = prompt_template.format(problem_statement=problem_statement, method_signature=method_signature)

    genai.configure(api_key=GOOGLE_API_KEY)
    model = genai.GenerativeModel("gemini-2.5-flash")

    response = model.generate_content(formatted_prompt)
    print(response.text)

    # Remove Markdown fences if present
    raw_response = response.text
    cleaned = raw_response.strip()
    if cleaned.startswith("```"):
        cleaned = "\n".join(cleaned.split("\n")[1:-1])  # remove first and last lines

    data = json.loads(cleaned)

    code_text = data["spec_list"]

    print(f"Successfully wrote spec_list for '{method_signature}'")

    return code_text

## Assignment Question 1

In [11]:
print("\ngemini scot\n")
res = test_suite("gemini_run_1","scot", CodeScore_df)
print(DataFrame(res))


gemini scot

             problem  #passed       line%     branch%       test%      metric
0            len_log        0  100.000000    0.000000    0.000000    0.000000
1     find_substring      102  100.000000  100.000000  100.000000  100.000000
2      is_undulating       85   93.103448   90.000000   83.333333   83.141762
3              power      102   75.000000   66.666667  100.000000   41.666667
4      index_minimum        0   14.285714    0.000000    0.000000    0.000000
5    Find_Min_Length      102   82.352941   66.666667  100.000000   49.019608
6            divisor      102   82.608696   80.000000  100.000000   62.608696
7    frequency_lists        0  100.000000    0.000000    0.000000    0.000000
8       multiply_num      102   83.333333   75.000000  100.000000   58.333333
9  decimal_to_binary        3   85.714286   75.000000    2.941176    4.640386


In [72]:
fml_text = CodeScore_df["text"][5]
power_text = CodeScore_df["text"][3]
fml_test = CodeScore_df["test_list"][5]
power_test = CodeScore_df["test_list"][3]
fml_method_signature = "Find_Min_Length(int[][] numlists) => int"
power_method_signature = "power(int base, int exponent) => int"

In [38]:
spec_gen_template = """
You are an expert Python programmer.
Write 5 formal specifications in the form of assertions (`assert` statements) for the following problem that describe the correct behavior of the method using the method's named parameters and "res" the returned value of the method. 

Problem: {problem_statement}

Method signature:{method_signature}

Notes: 
Do not call the function/method itself in your assertions. The only values should be "res" and the named parameters.
Do not use methods with side effects such as modifying data structures (add, append, remove, set), performing I/O (print, read, write, input, outputStream), using randomness or timing (Math.random(), System.currentTimeMillis(), datetime.now()).
Express the relationship between the method parameters and res using pure arithmetic and boolean logic only

Output solution in this exact JSON format:
{{
  "spec_list": Array of specifications using assert format.,
}}

Important:
- The 'code' must be valid Python.
- Do NOT include markdown fences (no ```python or ```).
- Keep the JSON strictly valid.
"""

In [45]:
fml_specs = gemini_generate_automated_specifications(spec_gen_template, problem_statement=fml_text, method_signature=fml_method_signature)

```json
{
  "spec_list": [
    "assert res >= 0",
    "assert len(numlists) == 0 or all(res <= len(sublist) for sublist in numlists)",
    "assert len(numlists) == 0 or any(len(sublist) == res for sublist in numlists)",
    "assert len(numlists) == 0 implies res == 0",
    "assert len(numlists) > 0 implies res == min(len(sublist) for sublist in numlists)"
  ]
}
```
Successfully wrote spec_list for 'power(int[][] numlists) => int'


In [46]:
power_specs = gemini_generate_automated_specifications(spec_gen_template, problem_statement=power_text, method_signature=power_method_signature)

```json
{
  "spec_list": [
    "assert exponent == 0 and res == 1",
    "assert exponent == 1 and res == base",
    "assert base == 1 and res == 1",
    "assert base == 0 and exponent > 0 and res == 0",
    "assert exponent == 2 and res == base * base"
  ]
}
```
Successfully wrote spec_list for 'power(int base, int exponent) => int'


In [64]:
new_fml_specs = fml_specs.copy()
new_fml_specs[3] = "assert len(numlists) != 0 or res == 0"
new_fml_specs[4] = "assert len(numlists)  <= 0 or res == min(len(sublist) for sublist in numlists)"
new_fml_specs

['assert res >= 0',
 'assert len(numlists) == 0 or all(res <= len(sublist) for sublist in numlists)',
 'assert len(numlists) == 0 or any(len(sublist) == res for sublist in numlists)',
 'assert len(numlists) != 0 or res == 0',
 'assert len(numlists)  <= 0 or res == min(len(sublist) for sublist in numlists)']

## Part 2

In [74]:
def gemini_generate_automated_tests_from_specs(prompt_template, problem_statement, method_signature, spec_list):
    import google.generativeai as genai

    formatted_prompt = prompt_template.format(problem_statement=problem_statement, method_signature=method_signature, spec_list=spec_list)

    genai.configure(api_key=GOOGLE_API_KEY)
    model = genai.GenerativeModel("gemini-2.5-flash")

    response = model.generate_content(formatted_prompt)
    print(response.text)

    # Remove Markdown fences if present
    raw_response = response.text
    cleaned = raw_response.strip()
    if cleaned.startswith("```"):
        cleaned = "\n".join(cleaned.split("\n")[1:-1])  # remove first and last lines

    data = json.loads(cleaned)

    code_text = data["test_list"]

    print(f"Successfully wrote tests for '{spec_list}'")

    return code_text

In [68]:
tests_from_spec_gen_template = """
You are an expert Python programmer.
Create tests for the following problem.
The tests should be based on the given specifications that describe the correct behavior of the method using true assertions about the method's named parameters and "res" the returned value of the method: 
 
Problem: {problem_statement}

Method signature: {method_signature}

Specifications: {spec_list}

Output solution in this exact JSON format:
{{
  "test_list: Array of tests using assert format.,
}}

Important:
- The 'code' must be valid Python.
- Do NOT include markdown fences (no ```python or ```).
- Keep the JSON strictly valid.
"""

In [73]:
fml_tests = gemini_generate_automated_tests_from_specs(tests_from_spec_gen_template, problem_statement=fml_text, method_signature=fml_method_signature, spec_list=new_fml_specs)

```json
{
  "test_list": [
    "assert Find_Min_Length([]) == 0",
    "assert Find_Min_Length([[]]) == 0",
    "assert Find_Min_Length([[], []]) == 0",
    "assert Find_Min_Length([[1, 2, 3], [], [4, 5]]) == 0",
    "assert Find_Min_Length([[1], [2, 3], [4, 5, 6]]) == 1",
    "assert Find_Min_Length([[1, 2], [3, 4], [5, 6]]) == 2",
    "assert Find_Min_Length([[1, 2, 3, 4]]) == 4",
    "assert Find_Min_Length([[1, 2, 3, 4], [5], [6, 7], [8, 9, 0]]) == 1",
    "assert Find_Min_Length([[1, 'a'], [True], [None, 2, 3]]) == 1",
    "assert Find_Min_Length([[10, 20, 30, 40, 50], [1, 2], [5, 6, 7, 8, 9, 10, 11]]) == 2"
  ]
}
```
Successfully wrote tests for '['assert res >= 0', 'assert len(numlists) == 0 or all(res <= len(sublist) for sublist in numlists)', 'assert len(numlists) == 0 or any(len(sublist) == res for sublist in numlists)', 'assert len(numlists) != 0 or res == 0', 'assert len(numlists)  <= 0 or res == min(len(sublist) for sublist in numlists)']'


In [85]:
power_tests = gemini_generate_automated_tests_from_specs(tests_from_spec_gen_template, problem_statement=power_text, method_signature=power_method_signature, spec_list=power_specs)

```json
{
  "test_list": [
    "assert power(5, 0) == 1",
    "assert power(-3, 0) == 1",
    "assert power(0, 0) == 1",
    "assert power(7, 1) == 7",
    "assert power(-2, 1) == -2",
    "assert power(0, 1) == 0",
    "assert power(1, 1) == 1",
    "assert power(1, 5) == 1",
    "assert power(1, 100) == 1",
    "assert power(0, 5) == 0",
    "assert power(0, 100) == 0",
    "assert power(3, 2) == 9",
    "assert power(-4, 2) == 16",
    "assert power(2, 3) == 8",
    "assert power(10, 3) == 1000",
    "assert power(-2, 3) == -8",
    "assert power(-2, 4) == 16",
    "assert power(2, 4) == 16"
  ]
}
```
Successfully wrote tests for '['assert exponent == 0 and res == 1', 'assert exponent == 1 and res == base', 'assert base == 1 and res == 1', 'assert base == 0 and exponent > 0 and res == 0', 'assert exponent == 2 and res == base * base']'


In [87]:
# unite the tests
combined_fml_tests = CodeScore_df['test_list'][5] + fml_tests
print(len(combined_fml_tests))

combined_power_tests = CodeScore_df['test_list'][3] + power_tests
print(len(combined_power_tests))

112
120


In [90]:
prev_test_list = CodeScore_df['test_list']
new_test_list = prev_test_list.copy()
new_test_list[5] = combined_fml_tests
new_test_list[3] = combined_power_tests
CodeScore_df['test_list'] = new_test_list

In [92]:
CodeScore_df['test_list'][5]

['assert Find_Min_Length([[1],[1,2]]) == 1',
 'assert Find_Min_Length([[1,2],[1,2,3],[1,2,3,4]]) == 2',
 'assert Find_Min_Length([[3,3,3],[4,4,4,4]]) == 3',
 'assert Find_Min_Length([[3], [6, 5]]) == 1',
 'assert Find_Min_Length([[6], [1, 4]]) == 1',
 'assert Find_Min_Length([[5], [3, 4]]) == 1',
 'assert Find_Min_Length([[4], [1, 7]]) == 1',
 'assert Find_Min_Length([[6], [2, 2]]) == 1',
 'assert Find_Min_Length([[2], [4, 1]]) == 1',
 'assert Find_Min_Length([[6], [1, 5]]) == 1',
 'assert Find_Min_Length([[5], [4, 5]]) == 1',
 'assert Find_Min_Length([[6], [2, 7]]) == 1',
 'assert Find_Min_Length([[3], [6, 6]]) == 1',
 'assert Find_Min_Length([[4], [5, 7]]) == 1',
 'assert Find_Min_Length([[1], [4, 1]]) == 1',
 'assert Find_Min_Length([[3], [3, 5]]) == 1',
 'assert Find_Min_Length([[6], [4, 1]]) == 1',
 'assert Find_Min_Length([[1], [5, 4]]) == 1',
 'assert Find_Min_Length([[1], [3, 7]]) == 1',
 'assert Find_Min_Length([[6], [1, 1]]) == 1',
 'assert Find_Min_Length([[4], [6, 6]]) == 1

In [93]:
print("\ngemini scot\n")
res = test_suite("gemini_run_1","scot", CodeScore_df)
print(DataFrame(res))


gemini scot

             problem  #passed       line%     branch%       test%      metric
0            len_log        0  100.000000    0.000000    0.000000    0.000000
1     find_substring      102  100.000000  100.000000  100.000000  100.000000
2      is_undulating       85   93.103448   90.000000   83.333333   83.141762
3              power      120   87.500000   83.333333  100.000000   70.833333
4      index_minimum        0   14.285714    0.000000    0.000000    0.000000
5    Find_Min_Length      111   94.117647   83.333333   99.107143   77.644339
6            divisor      102   82.608696   80.000000  100.000000   62.608696
7    frequency_lists        0  100.000000    0.000000    0.000000    0.000000
8       multiply_num      102   83.333333   75.000000  100.000000   58.333333
9  decimal_to_binary        3   85.714286   75.000000    2.941176    4.640386


In [94]:
run_tests_visible("gemini_run_1","selfrepair",5,combined_fml_tests)

✅ Passed: assert Find_Min_Length([[1],[1,2]]) == 1
✅ Passed: assert Find_Min_Length([[1,2],[1,2,3],[1,2,3,4]]) == 2
✅ Passed: assert Find_Min_Length([[3,3,3],[4,4,4,4]]) == 3
✅ Passed: assert Find_Min_Length([[3], [6, 5]]) == 1
✅ Passed: assert Find_Min_Length([[6], [1, 4]]) == 1
✅ Passed: assert Find_Min_Length([[5], [3, 4]]) == 1
✅ Passed: assert Find_Min_Length([[4], [1, 7]]) == 1
✅ Passed: assert Find_Min_Length([[6], [2, 2]]) == 1
✅ Passed: assert Find_Min_Length([[2], [4, 1]]) == 1
✅ Passed: assert Find_Min_Length([[6], [1, 5]]) == 1
✅ Passed: assert Find_Min_Length([[5], [4, 5]]) == 1
✅ Passed: assert Find_Min_Length([[6], [2, 7]]) == 1
✅ Passed: assert Find_Min_Length([[3], [6, 6]]) == 1
✅ Passed: assert Find_Min_Length([[4], [5, 7]]) == 1
✅ Passed: assert Find_Min_Length([[1], [4, 1]]) == 1
✅ Passed: assert Find_Min_Length([[3], [3, 5]]) == 1
✅ Passed: assert Find_Min_Length([[6], [4, 1]]) == 1
✅ Passed: assert Find_Min_Length([[1], [5, 4]]) == 1
✅ Passed: assert Find_Min_Leng